In [0]:
%pip install --upgrade setuptools
%restart_python

  Using cached setuptools-83.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-83.0.0-py3-none-any.whl (1.0 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 74.0.0
    Not uninstalling setuptools at /usr/local/lib/python3.12/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-af90110d-7676-4990-adef-8e21e11bb77b
    Can't uninstall 'setuptools'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pmdarima 2.1.1 requires statsmodels>=0.14.5, but you have statsmodels 0.14.4 which is incompatible.
spacy 3.8.14 requires confection<2.0.0,>=1.3.2, but you have confection 0.1.5 which is incompatible.
spacy 3.8.14 requires srsly<3.0.0,>=2.5.3, but you have srsly 2.5.1 which is incompatible.
spacy 3.8.14 requires weasel<2.0.0,>=1.0.0, but you have weasel 0.4.1 which is

In [0]:
import pandas as pd
import torch
from typing import Iterator
from transformers import AutoTokenizer, AutoModel
from pyspark.sql.functions import col, pandas_udf
from pyspark.sql.types import ArrayType, FloatType

spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "32")

2026-07-25 20:21:59.753624: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785010919.773016    4252 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785010919.779016    4252 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1785010919.794138    4252 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1785010919.794153    4252 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1785010919.794156    4252 computation_placer.cc:177] computation placer alr

[2026-07-25 20:22:02,013] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


In [ ]:
MODEL_PATH = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/resources/embedding-models/carina-embedding-3/carina-embedding-3.0/"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = AutoModel.from_pretrained(MODEL_PATH, local_files_only=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
@pandas_udf(ArrayType(FloatType()))
def generate_embeddings(iterator: Iterator[pd.Series]) -> Iterator[pd.Series]:

    # Model loaded inside the worker task, not the driver -- avoids a driver OOM.
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModel.from_pretrained(MODEL_PATH)
    model.to(device)

    for text_series in iterator:
        texts = text_series.fillna("").tolist()
        inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=512).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()

        del inputs
        del outputs
        torch.cuda.empty_cache()

        yield pd.Series(embeddings.tolist())

In [0]:
case_df = spark.sql("""
    SELECT case_id, case_description, case_solution_final_statement 
    FROM stg_ccc.w_case
    WHERE case_id IN (SELECT case_id FROM ds_ccc.kg_sulo_sample_cases_table)
      AND (case_description IS NOT NULL OR case_solution_final_statement IS NOT NULL)
""").repartition(1)

In [0]:
case_emb = case_df.withColumn("description_vector", generate_embeddings(col("case_description")))
case_emb = case_emb.withColumn("solution_vector", generate_embeddings(col("case_solution_final_statement")))

case_emb.select("case_id", "description_vector", "solution_vector").createOrReplaceTempView("temp_case_emb")
spark.sql("CREATE OR REPLACE TABLE ds_ccc.sulokg_embeddings_dv AS SELECT * FROM temp_case_emb")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [ ]:
task_df = spark.sql("""
    SELECT t.id AS task_id, t.whatid AS case_id, t.description AS task_description
    FROM westeurope_cac_dwh_prd.sf_ccc.task t
    WHERE t.whatid IN (SELECT case_id FROM ds_ccc.kg_sulo_sample_cases_table)
      AND t.description IS NOT NULL
""").repartition(1)

In [0]:
task_emb = task_df.withColumn("task_desc_vector", generate_embeddings(col("task_description")))

task_emb.select("task_id", "case_id", "task_desc_vector").createOrReplaceTempView("temp_task_emb")
spark.sql("CREATE OR REPLACE TABLE ds_ccc.sulokg_task_embeddings_dv AS SELECT * FROM temp_task_emb")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can